# AM5061 · Week 13 · ORC system convergence

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "glide", "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
    "lmtd", "effectiveness", "ntu_required", "exergy", "T0_REF", "P0_REF",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


# ------------------------------------------------- exchanger relations
def lmtd(dT1, dT2):
    """Log-mean temperature difference.

    Falls back to the arithmetic mean when the two ends are within 1% of each
    other, where the log form is numerically unstable and the two agree to
    better than 0.01% anyway.
    """
    dT1, dT2 = float(dT1), float(dT2)
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError(
            f"temperature difference must be positive at both ends "
            f"(got {dT1:g} and {dT2:g}). A non-positive end means the streams "
            "cross, which no exchanger of this configuration can do."
        )
    if abs(dT1 - dT2) < 0.01*max(dT1, dT2):
        return 0.5*(dT1 + dT2)
    return (dT1 - dT2)/math.log(dT1/dT2)


def effectiveness(config, NTU, Cr):
    """Effectiveness for the standard configurations.

    config: 'counter', 'parallel', 'shell1'  (one shell pass, 2/4/... tube passes),
            'cross-both-unmixed' (approximate), 'cross-Cmax-mixed', 'cross-Cmin-mixed'

    Cr = C_min/C_max. Cr = 0 is the phase-change limit and every configuration
    collapses to the same expression, which is why boilers and condensers are
    easy and everything else is not.
    """
    if NTU < 0:
        raise ValueError("NTU cannot be negative")
    if not 0 <= Cr <= 1:
        raise ValueError(f"Cr must be between 0 and 1, got {Cr:g}")
    if Cr == 0:                       # phase change on one side
        return 1 - math.exp(-NTU)
    if config == "counter":
        if abs(Cr - 1) < 1e-12:
            return NTU/(1 + NTU)
        e = math.exp(-NTU*(1 - Cr))
        return (1 - e)/(1 - Cr*e)
    if config == "parallel":
        return (1 - math.exp(-NTU*(1 + Cr)))/(1 + Cr)
    if config == "shell1":
        r = math.sqrt(1 + Cr*Cr)
        e = math.exp(-NTU*r)
        return 2/(1 + Cr + r*(1 + e)/(1 - e))
    if config == "cross-both-unmixed":
        return 1 - math.exp((math.exp(-Cr*NTU**0.78) - 1)*NTU**0.22/Cr)
    if config == "cross-Cmax-mixed":
        return (1/Cr)*(1 - math.exp(-Cr*(1 - math.exp(-NTU))))
    if config == "cross-Cmin-mixed":
        return 1 - math.exp(-(1 - math.exp(-Cr*NTU))/Cr)
    raise ValueError(f"unknown configuration {config!r}")


def ntu_required(config, eps, Cr, hi=200.0):
    """Invert effectiveness() for NTU. Design direction, rather than rating."""
    eps_max = effectiveness(config, hi, Cr)
    if eps >= eps_max:
        raise ValueError(
            f"effectiveness {eps:g} is unreachable for {config} at Cr={Cr:g}; "
            f"the limit as NTU->infinity is {eps_max:.6f}. "
            "Change the configuration or accept less."
        )
    return solve(lambda n: effectiveness(config, n, Cr) - eps, 1.0,
                 bracket=(1e-9, hi))


# --------------------------------------------------------------- exergy
T0_REF, P0_REF = 303.15, 101325.0      # 30 C, sea level: the Chennai dead state


def exergy(st, T0=T0_REF, p0=P0_REF):
    """Specific flow exergy, J/kg:  (h - h0) - T0*(s - s0).

    The dead state is the ambient the plant actually sits in, so it is a
    DESIGN CHOICE, not a constant. Report which one you used - a Chennai
    dead state and a European one give different answers for the same plant.
    """
    ref = State(st.fluid, T=T0, P=p0)
    return (st.h - ref.h) - T0*(st.s - ref.s)


---
## The case

An **Organic Rankine Cycle** recovering cement kiln waste heat. Six components,
four unknown state points, three constraints.

Deliverable **D-13**: optimise the **evaporation pressure** and the **working
fluid** for maximum net power. Submit the sweep and the optimum.

### The real subject this week

Not the ORC. **How a design code actually solves a coupled system**, and then
how it optimises one. Every case so far had a solve you could see. This one has
a loop you have to break deliberately.

The loop: the working-fluid flow depends on the **pinch point** in the
evaporator; the pinch depends on the temperature profile; the profile depends on
the flow. That is a fixed point, and how you attack it decides whether your code
converges, crawls, or diverges.


## 1. Degrees of freedom, counted before any code is written

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
from scipy.optimize import brentq, fsolve, minimize_scalar
am.style_plots()

print("""  Components:   pump, evaporator, turbine, condenser            (4)
  State points: 1 pump in, 2 pump out, 3 turbine in, 4 turbine out (4)

  Unknowns    : m_wf, and the four state points               
  Equations   : pump work, evaporator energy balance, turbine work,
                condenser energy balance                          (4)
  Specified   : condensing temperature, evaporation pressure,
                superheat, pump and turbine efficiencies

  Degrees of freedom left = 1  ->  the EVAPORATION PRESSURE.
  That is the single decision variable, and it is what we optimise.""")

# ---- the heat source ------------------------------------------------
m_gas, T_gas_in, T_gas_min = 2.0, am.K(300), am.K(120)   # kg/s, K, acid dew limit
def gas_cp(T): return PropsSI("C","P",101325,"T",T,"Air")
T_cond = am.K(40)
dT_pp  = 10.0        # pinch point in the evaporator
dT_sh  = 5.0         # superheat at turbine inlet
eta_t, eta_p = 0.80, 0.70
print(f"\n  gas {m_gas} kg/s at {am.C(T_gas_in):.0f} C, floor {am.C(T_gas_min):.0f} C")
print(f"  maximum recoverable = {m_gas*gas_cp(0.5*(T_gas_in+T_gas_min))*(T_gas_in-T_gas_min)/1e3:.1f} kW")


## 2. The loop, and two ways to break it

Guess the working-fluid flow, build the evaporator profile, find where the pinch
actually lands, and correct the flow. Repeat.

**Successive substitution** just feeds the new value back. It is trivial to
write and it diverges cheerfully. **Newton–Raphson** uses the local slope and
converges quadratically when it converges at all. Watch both.


In [ ]:
def orc_states(fluid, p_evap):
    """The four state points, given a working fluid and evaporation pressure."""
    p_cond = am.p_sat(fluid, T_cond)
    st1 = am.sat_liquid(fluid, p=p_cond)                       # pump inlet
    h2s = PropsSI("H","P",p_evap,"S",st1.s,fluid)
    h2  = st1.h + (h2s - st1.h)/eta_p                          # pump outlet
    T_ev = am.T_sat(fluid, p_evap)
    st3 = am.State(fluid, P=p_evap, T=T_ev + dT_sh)            # turbine inlet
    h4s = PropsSI("H","P",p_cond,"S",st3.s,fluid)
    h4  = st3.h - eta_t*(st3.h - h4s)                          # turbine outlet
    return {"p_cond": p_cond, "h1": st1.h, "h2": h2, "h3": st3.h, "h4": h4,
            "T_evap": T_ev, "st3": st3}

def flow_from_pinch(fluid, p_evap, m_guess):
    """Given a flow guess, where does the pinch land and what flow does that
    imply? This is the fixed-point map g(m)."""
    s = orc_states(fluid, p_evap)
    # Gas cp is evaluated at the ACTUAL mean gas temperature, which depends on
    # how much heat the cycle takes, which depends on the flow we are solving
    # for. THAT is what makes this a genuine fixed point rather than a formula.
    Q_total = m_guess*(s["h3"] - s["h2"])
    cp_first = gas_cp(0.5*(T_gas_in + T_gas_min))
    T_gas_out_est = T_gas_in - Q_total/(m_gas*cp_first)
    cp_g = gas_cp(0.5*(T_gas_in + max(T_gas_out_est, am.K(60))))
    # gas temperature at the point where the working fluid starts to boil
    h_f = PropsSI("H","P",p_evap,"Q",0,fluid)
    Q_sup_evap = m_guess*(s["h3"] - h_f)                # boiling + superheat duty
    T_gas_pinch = T_gas_in - Q_sup_evap/(m_gas*cp_g)
    # the pinch constraint says that gas temperature must sit dT_pp above T_evap
    T_gas_pinch_req = s["T_evap"] + dT_pp
    # correct the flow so the constraint is met
    return (T_gas_in - T_gas_pinch_req)*m_gas*cp_g/(s["h3"] - h_f), T_gas_pinch

def solve_successive(fluid, p_evap, m0=1.0, tol=1e-10, itmax=200, damp=1.0):
    m, hist = m0, [m0]
    for i in range(itmax):
        m_new, _ = flow_from_pinch(fluid, p_evap, m)
        m_next = m + damp*(m_new - m)
        hist.append(m_next)
        if abs(m_next - m) < tol: return m_next, hist, True
        m = m_next
    return m, hist, False

def solve_newton(fluid, p_evap, m0=1.0):
    f = lambda m: flow_from_pinch(fluid, p_evap, m)[0] - m
    hist = []
    def g(m):
        hist.append(float(m[0]) if hasattr(m, "__len__") else float(m))
        return f(hist[-1])
    sol = fsolve(g, m0, full_output=False)
    return float(sol[0]), hist, True
print("  solvers defined")


In [ ]:
FL = "R245fa"
p_ev = 15e5
m_ss, h_ss, ok_ss = solve_successive(FL, p_ev, m0=1.0)
m_nw, h_nw, _     = solve_newton(FL, p_ev, m0=1.0)
print(f"  successive substitution: m = {m_ss:.6f} kg/s in {len(h_ss)-1} iterations"
      f"  ({'converged' if ok_ss else 'DID NOT CONVERGE'})")
print(f"  Newton-Raphson         : m = {m_nw:.6f} kg/s in {len(h_nw)} evaluations")
print(f"  agreement: {abs(m_ss-m_nw):.2e} kg/s")

fig, ax = plt.subplots()
ax.plot(range(len(h_ss)), h_ss, "o-", lw=2, label="successive substitution")
ax.plot(range(len(h_nw)), h_nw, "s-", lw=2, label="Newton (fsolve)")
ax.axhline(m_nw, color=am.MUTED, ls="--")
ax.set_xlabel("iteration"); ax.set_ylabel("working fluid flow  (kg/s)")
ax.set_xlim(0, min(25, max(len(h_ss), len(h_nw))))
ax.set_title("Same answer, very different paths"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


### Damping

If successive substitution oscillates or diverges, the standard fix is
**under-relaxation**: take only a fraction of the proposed step. It costs
iterations and buys stability. This is exactly what a plant code does when it
will not converge.


In [ ]:
print(f"{'damping':>9}{'iterations':>13}{'converged':>12}")
for d in (1.5, 1.0, 0.7, 0.4, 0.2):
    m_, h_, ok_ = solve_successive(FL, p_ev, m0=1.0, damp=d)
    print(f"{d:9.1f}{len(h_)-1:13d}{str(ok_):>12}")
print("\n  Read that carefully: damping = 1.0 (no damping) is FASTEST here, at 3")
print("  iterations. Both over-relaxing and under-relaxing make it worse.")
print("  Damping is insurance, not acceleration: you pay iterations for it, and")
print("  you only pay willingly on a problem that will not converge without it.")


## 3. Optimising the evaporation pressure

In [ ]:
def performance(fluid, p_evap):
    """Net power for one fluid at one evaporation pressure."""
    try:
        p_c = am.critical(fluid)["p"]
        if p_evap > 0.95*p_c: return None
        s = orc_states(fluid, p_evap)
        m_wf, _, _ = solve_newton(fluid, p_evap, m0=1.0)
        if m_wf <= 0: return None
        cp_g = gas_cp(0.5*(T_gas_in + T_gas_min))
        Q_in = m_wf*(s["h3"] - s["h2"])
        T_gas_out = T_gas_in - Q_in/(m_gas*cp_g)
        # The acid dew floor is a CONSTRAINT, not a disqualification. If the
        # pinch-limited flow would over-cool the stack, throttle the flow until
        # the stack sits exactly on the floor. Which constraint binds is then
        # itself a design finding.
        binding = "pinch"
        if T_gas_out < T_gas_min:
            Q_in = m_gas*cp_g*(T_gas_in - T_gas_min)
            m_wf = Q_in/(s["h3"] - s["h2"])
            T_gas_out = T_gas_min
            binding = "stack floor"
        W_t = m_wf*(s["h3"] - s["h4"]); W_p = m_wf*(s["h2"] - s["h1"])
        return {"fluid": fluid, "binding constraint": binding, "p_evap, bar": p_evap/1e5,
                "T_evap, C": am.C(s["T_evap"]), "m_wf, kg/s": m_wf,
                "Q_in, kW": Q_in/1e3, "W_turbine, kW": W_t/1e3,
                "W_pump, kW": W_p/1e3, "W_net, kW": (W_t-W_p)/1e3,
                "eta_thermal": (W_t-W_p)/Q_in,
                "T_gas_out, C": am.C(T_gas_out),
                "x_turbine_exit": am.State(fluid,P=s["p_cond"],H=s["h4"]).x}
    except Exception as e:
        # Report the first failure rather than silently returning None for
        # everything, which is how a whole sweep can come back empty.
        if not performance._warned:
            print(f"    [performance] first failure on {fluid} at "
                  f"{p_evap/1e5:.1f} bar: {type(e).__name__}: {e}")
            performance._warned = True
        return None
performance._warned = False

FLUIDS = ["R245fa", "R1233zd(E)", "n-Pentane", "IsoButane", "Toluene", "R1336mzz(Z)"]
best = {}
fig, ax = plt.subplots()
for fl in FLUIDS:
    p_c = am.critical(fl)["p"]
    ps  = np.linspace(3e5, 0.92*p_c, 60)
    pts = [(p, performance(fl, float(p))) for p in ps]
    pts = [(p, r_) for p, r_ in pts if r_]
    if not pts: continue
    ax.plot([p/1e5 for p, _ in pts], [r_["W_net, kW"] for _, r_ in pts], lw=2.2, label=fl)
    bp, br = max(pts, key=lambda t: t[1]["W_net, kW"])
    best[fl] = br
    ax.plot(bp/1e5, br["W_net, kW"], "o", ms=8, color=am.NAVY, zorder=5)
ax.set_xlabel("evaporation pressure  (bar)"); ax.set_ylabel("net power  (kW)")
ax.set_title("One decision variable, one optimum per fluid"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f"{'fluid':>13}{'p_opt bar':>11}{'T_evap C':>10}{'W_net kW':>10}"
      f"{'eta_th':>9}{'T_gas_out':>11}{'x_exit':>9}{'binds':>13}")
for fl, r_ in sorted(best.items(), key=lambda t: -t[1]["W_net, kW"]):
    x = r_["x_turbine_exit"]
    xs = f"{x:9.3f}" if 0 <= x <= 1 else f"{'dry':>9}"
    print(f"{fl:>13}{r_['p_evap, bar']:11.2f}{r_['T_evap, C']:10.1f}"
          f"{r_['W_net, kW']:10.2f}{r_['eta_thermal']:9.4f}"
          f"{r_['T_gas_out, C']:11.1f}{xs}{r_['binding constraint']:>13}")


### Every fluid binds on the same constraint

Look at the last column. None of these designs is limited by the evaporator
pinch: they are all limited by the **acid dew floor** on the stack. That is a
finding, not a detail. It means the exchanger is not the bottleneck, the
chemistry of the flue gas is, and buying more heat transfer area would buy
nothing at all.

### Reading the table

`x_exit` matters as much as the power. A fluid that expands **wet** erodes the
turbine; "dry" means the expansion ends superheated, which is why ORC designers
prefer **dry** fluids (those with a positive-slope saturated vapour line) and
why the highest-power fluid is not automatically the right choice.

Check `T_gas_out` too: a design that beats the field on paper but drives the
stack below the acid dew point is not a design.


## 4. Refined optimum, and the deliverable

In [ ]:
rows = []
for fl in FLUIDS:
    p_c = am.critical(fl)["p"]
    def neg(p):
        q = performance(fl, float(p))
        return -q["W_net, kW"] if q else 1e9
    res = minimize_scalar(neg, bounds=(3e5, 0.92*p_c), method="bounded")
    r_ = performance(fl, float(res.x))
    if r_: rows.append(r_)
rows.sort(key=lambda r_: -r_["W_net, kW"])
if not rows:
    raise RuntimeError("no fluid produced a feasible design - check the pinch "
                       "and the acid dew floor before going further")
win = rows[0]
print(f"  optimum overall: {win['fluid']} at {win['p_evap, bar']:.2f} bar, "
      f"{win['W_net, kW']:.2f} kW net, eta_th {win['eta_thermal']:.4f}")

sweep = []
for fl in FLUIDS:
    p_c = am.critical(fl)["p"]
    for p in np.linspace(3e5, 0.92*p_c, 30):
        r_ = performance(fl, float(p))
        if r_: sweep.append(r_)

path = am.to_excel("AM5061_D13_ORC.xlsx",
    {"Optimum per fluid": rows, "Full sweep": sweep},
    title="AM5061 D-13 . ORC on cement kiln waste heat",
    summary=[("Gas flow", m_gas, "kg/s"), ("Gas inlet", am.C(T_gas_in), "C"),
             ("Gas floor (acid dew)", am.C(T_gas_min), "C"),
             ("Condensing temperature", am.C(T_cond), "C"),
             ("Evaporator pinch", dT_pp, "K"),
             ("Turbine / pump efficiency", f"{eta_t} / {eta_p}", "-"),
             ("Best fluid", win["fluid"], ""),
             ("Optimum evaporation pressure", win["p_evap, bar"], "bar"),
             ("Net power at optimum", win["W_net, kW"], "kW"),
             ("Thermal efficiency", win["eta_thermal"], "-")],
    sources=[("Working fluid properties", "CoolProp 8.0.0"),
             ("Flue gas", "modelled as air"),
             ("Solution method", "Newton-Raphson on the evaporator pinch fixed point"),
             ("Case data", "AM5061 brief D-13")])
print("written:", path)


## What to hand in

1. The convergence comparison: successive substitution against Newton, with the
   iteration counts and the damping study.
2. The degrees-of-freedom count, done **before** the code.
3. Net power against evaporation pressure for every fluid, with the optima
   marked.
4. Your chosen fluid and pressure, justified on **power, expansion dryness and
   stack temperature together** — not power alone.
5. The workbook.

**One paragraph:** your optimiser found a maximum. How do you know it is not a
local one, and what would you do to convince a reviewer?
